## Query raw data and save to bucket

In [ ]:
# Query ADHD cohort - demographics, ADHD and dependence dx data, zip code SES data

library(tidyverse)
library(bigrquery)

# This query represents dataset "ADHD WGS EHR - demographics only" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_08837079_person_sql <- paste("
    SELECT
        person.person_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        p_race_concept.concept_name as race,
        p_ethnicity_concept.concept_name as ethnicity,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (4149353, 4149904, 438409, 4253962, 40480225)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        concept_id IN (44822997, 35207264, 44829951, 35207262, 45552506, 35207263, 35207265) 
                        AND is_standard = 0 )) criteria ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_08837079_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person_08837079",
  "person_08837079_*.csv")
message(str_glue('The data will be written to {person_08837079_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_08837079_person_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  person_08837079_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_08837079_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(gender = col_character(), race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_08837079_person_df <- read_bq_export_from_workspace_bucket(person_08837079_path)

dim(dataset_08837079_person_df)

head(dataset_08837079_person_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "ADHD WGS EHR - demographics only" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v8
dataset_08837079_zip_code_socioeconomic_sql <- paste("
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `zip3_ses_map` zip_code 
    JOIN
        `observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (4149353, 4149904, 438409, 4253962, 40480225)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        concept_id IN (44822997, 35207264, 44829951, 35207262, 45552506, 35207263, 35207265) 
                        AND is_standard = 0 )) criteria ) ) 
                    AND observation_source_concept_id = 1585250 
                    AND observation.value_as_string NOT LIKE 'Res%'", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
zip_code_socioeconomic_08837079_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "zip_code_socioeconomic_08837079",
  "zip_code_socioeconomic_08837079_*.csv")
message(str_glue('The data will be written to {zip_code_socioeconomic_08837079_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_08837079_zip_code_socioeconomic_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  zip_code_socioeconomic_08837079_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {zip_code_socioeconomic_08837079_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(zip3_as_string = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_08837079_zip_code_socioeconomic_df <- read_bq_export_from_workspace_bucket(zip_code_socioeconomic_08837079_path)

dim(dataset_08837079_zip_code_socioeconomic_df)

head(dataset_08837079_zip_code_socioeconomic_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "ADHD WGS EHR - demographics only" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_08837079_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.stop_reason,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (37017563, 37018356, 37110407, 37110436, 40480225, 4099811, 4149353, 4149904, 4209423, 4218106, 4253962, 433452, 433994, 435243, 436389, 437264, 437838, 438120, 438409, 440387, 440692, 440693)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1) 
                OR  condition_source_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (35207262, 35207263, 35207264, 35207265, 44822997, 44829951, 45552506)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 0 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `cb_criteria` cr       
                                    WHERE
                                        concept_id IN (4149353, 4149904, 438409, 4253962, 40480225)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `cb_search_all_events` 
                            WHERE
                                concept_id IN (44822997, 35207264, 44829951, 35207262, 45552506, 35207263, 35207265) 
                                AND is_standard = 0 )) criteria ) )
                        )
                    ) c_occurrence 
                LEFT JOIN
                    `concept` c_standard_concept 
                        ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
                LEFT JOIN
                    `visit_occurrence` v 
                        ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
                LEFT JOIN
                    `concept` visit 
                        ON v.visit_concept_id = visit.concept_id 
                LEFT JOIN
                    `concept` c_source_concept 
                        ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_08837079_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_08837079",
  "condition_08837079_*.csv")
message(str_glue('The data will be written to {condition_08837079_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_08837079_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_08837079_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_08837079_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), stop_reason = col_character(), visit_occurrence_concept_name = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_08837079_condition_df <- read_bq_export_from_workspace_bucket(condition_08837079_path)

dim(dataset_08837079_condition_df)

head(dataset_08837079_condition_df, 5)

In [ ]:
# Query NO ADHD cohort
library(tidyverse)
library(bigrquery)

# This query represents dataset "011626 NO ADHD cohort" for domain "zip_code_socioeconomic" and was generated for All of Us Controlled Tier Dataset v8
dataset_60152030_zip_code_socioeconomic_sql <- paste("
    SELECT
        observation.person_id,
        observation.observation_datetime,
        zip_code.zip3_as_string as zip_code,
        zip_code.fraction_assisted_income as assisted_income,
        zip_code.fraction_high_school_edu as high_school_education,
        zip_code.median_income,
        zip_code.fraction_no_health_ins as no_health_insurance,
        zip_code.fraction_poverty as poverty,
        zip_code.fraction_vacant_housing as vacant_housing,
        zip_code.deprivation_index,
        zip_code.acs as american_community_survey_year 
    FROM
        `zip3_ses_map` zip_code 
    JOIN
        `observation` observation 
            ON CAST(SUBSTR(observation.value_as_string, 0, STRPOS(observation.value_as_string, '*') - 1) AS INT64) = zip_code.zip3  
    WHERE
        observation.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (4149353, 4149904, 438409, 4253962, 40480225)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        concept_id IN (44822997, 35207264, 44829951, 35207262, 45552506, 35207263, 35207265) 
                        AND is_standard = 0 )) criteria ) ) 
                    AND observation_source_concept_id = 1585250 
                    AND observation.value_as_string NOT LIKE 'Res%'", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
zip_code_socioeconomic_60152030_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "zip_code_socioeconomic_60152030",
  "zip_code_socioeconomic_60152030_*.csv")
message(str_glue('The data will be written to {zip_code_socioeconomic_60152030_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_60152030_zip_code_socioeconomic_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  zip_code_socioeconomic_60152030_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {zip_code_socioeconomic_60152030_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(zip3_as_string = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_60152030_zip_code_socioeconomic_df <- read_bq_export_from_workspace_bucket(zip_code_socioeconomic_60152030_path)

dim(dataset_60152030_zip_code_socioeconomic_df)

head(dataset_60152030_zip_code_socioeconomic_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "011626 NO ADHD cohort" for domain "person" and was generated for All of Us Controlled Tier Dataset v8
dataset_60152030_person_sql <- paste("
    SELECT
        person.person_id,
        p_gender_concept.concept_name as gender,
        person.birth_datetime as date_of_birth,
        p_race_concept.concept_name as race,
        p_ethnicity_concept.concept_name as ethnicity,
        p_sex_at_birth_concept.concept_name as sex_at_birth,
        p_self_reported_category_concept.concept_name as self_reported_category 
    FROM
        `person` person 
    LEFT JOIN
        `concept` p_gender_concept 
            ON person.gender_concept_id = p_gender_concept.concept_id 
    LEFT JOIN
        `concept` p_race_concept 
            ON person.race_concept_id = p_race_concept.concept_id 
    LEFT JOIN
        `concept` p_ethnicity_concept 
            ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
    LEFT JOIN
        `concept` p_sex_at_birth_concept 
            ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
    LEFT JOIN
        `concept` p_self_reported_category_concept 
            ON person.self_reported_category_concept_id = p_self_reported_category_concept.concept_id  
    WHERE
        person.PERSON_ID IN (SELECT
            distinct person_id  
        FROM
            `cb_search_person` cb_search_person  
        WHERE
            cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_whole_genome_variant = 1 
            UNION
            DISTINCT SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_lr_whole_genome_variant = 1 ) 
            AND cb_search_person.person_id IN (SELECT
                person_id 
            FROM
                `cb_search_person` p 
            WHERE
                has_ehr_data = 1 ) 
            AND cb_search_person.person_id NOT IN (SELECT
                criteria.person_id 
            FROM
                (SELECT
                    DISTINCT person_id, entry_date, concept_id 
                FROM
                    `cb_search_all_events` 
                WHERE
                    person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        concept_id IN(SELECT
                            DISTINCT c.concept_id 
                        FROM
                            `cb_criteria` c 
                        JOIN
                            (SELECT
                                CAST(cr.id as string) AS id       
                            FROM
                                `cb_criteria` cr       
                            WHERE
                                concept_id IN (4149353, 4149904, 438409, 4253962, 40480225)       
                                AND full_text LIKE '%_rank1]%'      ) a 
                                ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                OR c.path LIKE CONCAT('%.', a.id) 
                                OR c.path LIKE CONCAT(a.id, '.%') 
                                OR c.path = a.id) 
                        WHERE
                            is_standard = 1 
                            AND is_selectable = 1) 
                        AND is_standard = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `cb_search_all_events` 
                    WHERE
                        concept_id IN (44822997, 35207264, 44829951, 35207262, 45552506, 35207263, 35207265) 
                        AND is_standard = 0 )) criteria ) )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
person_60152030_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "person_60152030",
  "person_60152030_*.csv")
message(str_glue('The data will be written to {person_60152030_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_60152030_person_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  person_60152030_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {person_60152030_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(gender = col_character(), race = col_character(), ethnicity = col_character(), sex_at_birth = col_character(), self_reported_category = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_60152030_person_df <- read_bq_export_from_workspace_bucket(person_60152030_path)

dim(dataset_60152030_person_df)

head(dataset_60152030_person_df, 5)
library(tidyverse)
library(bigrquery)

# This query represents dataset "011626 NO ADHD cohort" for domain "condition" and was generated for All of Us Controlled Tier Dataset v8
dataset_60152030_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime,
        c_occurrence.condition_end_datetime,
        c_occurrence.visit_occurrence_id,
        visit.concept_name as visit_occurrence_concept_name,
        c_occurrence.condition_source_value,
        c_source_concept.concept_name as source_concept_name,
        c_source_concept.concept_code as source_concept_code,
        c_source_concept.vocabulary_id as source_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (37017563, 37018356, 37110407, 37110436, 4099811, 4209423, 4218106, 433452, 433994, 435243, 436389, 437264, 437838, 438120, 440387, 440692, 440693)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 
                    UNION
                    DISTINCT SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_lr_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) 
                    AND cb_search_person.person_id NOT IN (SELECT
                        criteria.person_id 
                    FROM
                        (SELECT
                            DISTINCT person_id, entry_date, concept_id 
                        FROM
                            `cb_search_all_events` 
                        WHERE
                            person_id IN (SELECT
                                person_id 
                            FROM
                                `cb_search_all_events` 
                            WHERE
                                concept_id IN(SELECT
                                    DISTINCT c.concept_id 
                                FROM
                                    `cb_criteria` c 
                                JOIN
                                    (SELECT
                                        CAST(cr.id as string) AS id       
                                    FROM
                                        `cb_criteria` cr       
                                    WHERE
                                        concept_id IN (4149353, 4149904, 438409, 4253962, 40480225)       
                                        AND full_text LIKE '%_rank1]%'      ) a 
                                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                                        OR c.path LIKE CONCAT('%.', a.id) 
                                        OR c.path LIKE CONCAT(a.id, '.%') 
                                        OR c.path = a.id) 
                                WHERE
                                    is_standard = 1 
                                    AND is_selectable = 1) 
                                AND is_standard = 1 
                            UNION
                            DISTINCT SELECT
                                person_id 
                            FROM
                                `cb_search_all_events` 
                            WHERE
                                concept_id IN (44822997, 35207264, 44829951, 35207262, 45552506, 35207263, 35207265) 
                                AND is_standard = 0 )) criteria ) )
                        )
                    ) c_occurrence 
                LEFT JOIN
                    `concept` c_standard_concept 
                        ON c_occurrence.condition_concept_id = c_standard_concept.concept_id 
                LEFT JOIN
                    `visit_occurrence` v 
                        ON c_occurrence.visit_occurrence_id = v.visit_occurrence_id 
                LEFT JOIN
                    `concept` visit 
                        ON v.visit_concept_id = visit.concept_id 
                LEFT JOIN
                    `concept` c_source_concept 
                        ON c_occurrence.condition_source_concept_id = c_source_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_60152030_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_60152030",
  "condition_60152030_*.csv")
message(str_glue('The data will be written to {condition_60152030_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_60152030_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_60152030_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_60152030_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character(), visit_occurrence_concept_name = col_character(), condition_source_value = col_character(), source_concept_name = col_character(), source_concept_code = col_character(), source_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
dataset_60152030_condition_df <- read_bq_export_from_workspace_bucket(condition_60152030_path)

dim(dataset_60152030_condition_df)

head(dataset_60152030_condition_df, 5)

In [ ]:
# Confirm saved to bucket

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# List objects in the bucket
system(paste0("gsutil ls -r ", my_bucket), intern=T)

### Raw data filepaths
'gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_adhd_condition_df.csv'
'gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_adhd_person_df.csv'
'gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_adhd_zip_df.csv'
'gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_control_condition_df.csv'
'gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_control_person_df.csv'
'gs://fc-secure-7ae80de6-52d1-43da-b022-1de783982d19/data/011626_control_zip_df.csv'

## Process ADHD cohort for inclusion/exclusion criteria

In [ ]:
library(tidyverse)
## Load ADHD condition data

# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- '011626_adhd_condition_df.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
adhd_condition_df  <- read_csv(name_of_file_in_bucket)
head(adhd_condition_df)

In [ ]:
unique(adhd_condition_df$standard_concept_name)

In [ ]:
# Define ADHD diagnosis codes to retain
dx_list_ADHD <- c(
  "Attention deficit hyperactivity disorder, combined type",
  "Attention deficit hyperactivity disorder, predominantly inattentive type",
  "Attention deficit hyperactivity disorder, predominantly hyperactive impulsive type",
  "Attention deficit hyperactivity disorder",
  "Adult attention deficit hyperactivity disorder",
  "Child attention deficit disorder",
  "Undifferentiated attention deficit disorder"
)

# Identify individuals with ADHD diagnosis in 2013 or later
person_list_adhd <- adhd_condition_df %>%
  select(person_id, standard_concept_name, condition_start_datetime) %>%
  distinct() %>%
  filter(standard_concept_name %in% dx_list_ADHD) %>%
  mutate(condition_start_date = year(condition_start_datetime)) %>%
  filter(condition_start_date >= 2013)

cat("Number of individuals with a post-2013 ADHD diagnosis:")
person_list_adhd %>%
  select(person_id) %>%
  n_distinct()

In [ ]:
# Filter person_list_adhd to exclude individuals with fewer than 2 recorded occurrences for ADHD

person_list_adhd <- person_list_adhd %>%
    mutate(n_occurrences = n_distinct(condition_start_datetime),
             .by = person_id) %>%
    filter(n_occurrences >= 2)

head(person_list_adhd)

cat("Number of individuals with 2 or more qualifying ADHD dx occurrences:")
person_list_adhd %>%
    select(person_id) %>%
    n_distinct()

In [ ]:
# Finalize person_list_adhd - just a vector of person ids now
person_list_adhd <- person_list_adhd %>%
    select(person_id) %>%
    unique()
head(person_list_adhd)

In [ ]:
# Filter adhd_condition_df for qualifying cases
filtered_adhd_condition_df <- adhd_condition_df %>%
    filter(person_id %in% person_list_adhd$person_id)
head(filtered_adhd_condition_df)

cat("Number of qualifying ADHD cases:")
n_distinct(filtered_adhd_condition_df$person_id)

In [ ]:
## Load person and zip data for ADHD cohort
cat("Loading person and zip data for ADHD cohort...")
# Person

    # replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
    name_of_file_in_bucket <- '011626_adhd_person_df.csv'

    # Get the bucket name
    my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

    # Copy the file from current workspace to the bucket
    system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

    # Load the file into a dataframe
    adhd_person_df  <- read_csv(name_of_file_in_bucket)
    head(adhd_person_df)

# Zip

    # replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
    name_of_file_in_bucket <- '011626_adhd_zip_df.csv'

    # Get the bucket name
    my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

    # Copy the file from current workspace to the bucket
    system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

    # Load the file into a dataframe
    adhd_zip_df  <- read_csv(name_of_file_in_bucket)
    head(adhd_zip_df)

cat("Person and zip data loaded.")

In [ ]:
# Filter adhd_person_df and adhd_zip_df for qualifying cases
cat("Filtering for qualifying cases...\n")
filtered_adhd_person_df <- adhd_person_df %>%
    filter(person_id %in% person_list_adhd$person_id)

filtered_adhd_zip_df <- adhd_zip_df %>%
    filter(person_id %in% person_list_adhd$person_id)
cat("Filtered adhd_person_df and adhd_zip df for qualifying cases.\n")
cat("CHECK: Number of cases in filtered_adhd_person_df:")
n_distinct(filtered_adhd_person_df$person_id)

## Case control matching

### Process data

In [ ]:
## Load control cohort person data

# replace 'test.csv' with the name of the file in your google bucket (don't delete the quotation marks)
name_of_file_in_bucket <- '011626_control_person_df.csv'

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ", my_bucket, "/data/", name_of_file_in_bucket, " ."), intern=T)

# Load the file into a dataframe
control_person_df  <- read_csv(name_of_file_in_bucket)
head(control_person_df)

In [ ]:
## Add group columns

filtered_adhd_person_df <- filtered_adhd_person_df %>%
    mutate(exposure = 1L,
           group = "ADHD")

control_person_df <- control_person_df %>%
    mutate(exposure = 0L,
           group = "Control")

In [ ]:
nrow(control_person_df)

In [ ]:
## Bind rows

person_df <- bind_rows(filtered_adhd_person_df, control_person_df)
n_distinct(person_df$person_id)

In [ ]:
## Define function: clean_person_df()
  clean_person_df <- function(data) {

  data %>%
    select(person_id, gender, date_of_birth, race, 
             ethnicity, sex_at_birth, self_reported_category, group, exposure) %>%
    mutate(
      # Recode gender
      gender = case_match(gender,
                          "Gender Identity: Non Binary" ~ "Nonbinary",
                          "Female" ~ "Female",
                          "Male" ~ "Male",
                          "PMI: Skip" ~ "Skipped",
                          "Gender Identity: Transgender" ~ "Transgender",
                          "Not man only, not woman only, prefer not to answer, or skipped" ~ "Skipped",
                          "Gender Identity: Additional Options" ~ "Other",
                          "I prefer not to answer" ~ "Skipped",
                          .default = "ERROR"),
      
      # Recode race
      race = case_match(race,
                        "None of these" ~ "Other",
                        "I prefer not to answer" ~ "Skipped",
                        "Middle Eastern or North African" ~ "MENA",
                        "Asian" ~ "Asian",
                        "PMI: Skip" ~ "Skipped",
                        "Native Hawaiian or Other Pacific Islander" ~ "Native Hawaiian/Pacific Islander",
                        "American Indian or Alaska Native" ~ "American Indian/Alaska Native",
                        "More than one population" ~ "Multiple",
                        "White" ~ "White",
                        "Black or African American" ~ "Black",
                        "None Indicated" ~ "Skipped",
                        .default = "ERROR"),
    
      # Recode ethnicity
      ethnicity = case_match(
        ethnicity,
        "What Race Ethnicity: Race Ethnicity None Of These" ~ "Other",
        "PMI: Prefer Not To Answer" ~ "Skipped",
        "Not Hispanic or Latino" ~ "Non-Hispanic/Latino",
        "PMI: Skip" ~ "Skipped",
        "Hispanic or Latino" ~ "Hispanic/Latino",
        .default = "ERROR"),
    
      # Recode sex at birth
      sex_at_birth = case_match(
        sex_at_birth,
        "Intersex" ~ "Intersex",
        "Female" ~ "Female",
        "Male" ~ "Male",
        "I prefer not to answer" ~ "Skipped",
        "PMI: Skip" ~ "Skipped",
        "Sex At Birth: Sex At Birth None Of These" ~ "Other",
        .default = "ERROR"),
    
      # Recode self reported category
      self_reported_category = case_match(
        self_reported_category,
        "None of these" ~ "Other",
        "I prefer not to answer" ~ "Skipped",
        "Middle Eastern or North African" ~ "MENA",
        "Asian" ~ "Asian",
        "PMI: Skip" ~ "Skipped",
        "Native Hawaiian or Other Pacific Islander" ~ "Native Hawaiian/Pacific Islander",
        "American Indian or Alaska Native" ~ "American Indian/Alaska Native",
        "More than one population" ~ "Multiple",
        "Black or African American" ~ "Black",
        "What Race Ethnicity: Hispanic" ~ "Hispanic",
        "White" ~ "White",
        .default = "ERROR"),
    
      # Create birth_year from date_of_birth
      birth_year = year(date_of_birth)
    ) %>%
  select(-date_of_birth)
}

In [ ]:
## Clean person_df
person_clean <- clean_person_df(person_df)

In [ ]:
head(person_clean)

In [ ]:
table(person_clean$group)

In [ ]:
## Exclude individuals with sex_at_birth other than Male or Female (PheWAS requirement)
# Male = 1 | Female = 0

person_match <- person_clean %>%
    filter(sex_at_birth == 'Male'|
           sex_at_birth == 'Female') %>%
    mutate(sex_binary = case_when(
            sex_at_birth == 'Male' ~ 1L,
            sex_at_birth == 'Female' ~ 0L))
head(person_match)
table(person_match$sex_at_birth, person_match$group)

### Case-control (propensity score) matching

In [ ]:
install.packages('MatchIt')
library(MatchIt)

In [ ]:
# Execute case-control matching
# 1:4 case control ratio
# Match cases and controls based on sex at birth, birth year, and self-reported race/ethnicity
demographics <- matchit(
  exposure ~ sex_at_birth + birth_year + self_reported_category,
  data    = person_match,
  method  = "nearest",
  exact   = ~sex_at_birth,
  ratio   = 4,
  replace = FALSE
)

# Visualize matching quality
plot(demographics, type = "density")

# Extract matched dataset
demographics <- match.data(demographics)

In [ ]:
## Save to bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- demographics

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- '011626_demographics.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)


## PheWAS performed in notebook titled 011626 PheTK.